In [ ]:
!pip -q install gensim POT

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gensim.downloader as api
from gensim.parsing.preprocessing import STOPWORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
glove = api.load("glove-wiki-gigaword-50")
print("GloVe loaded.")

In [ ]:
s1 = "Obama speaks to the media in Illinois"
s2 = "The president greets the press in Chicago"
s3 = "Oranges are my favorite fruit"
s4 = "The team won the football match yesterday"
sents = [s1, s2, s3, s4]

def prep(s):
    return [w for w in s.lower().split() if w not in STOPWORDS and w in glove]

for s in sents: print(prep(s))

In [ ]:
print("WMD(s1, s2) =", round(glove.wmdistance(prep(s1), prep(s2)), 4), " <- similar meaning")
print("WMD(s1, s3) =", round(glove.wmdistance(prep(s1), prep(s3)), 4), " <- unrelated")
print("WMD(s1, s4) =", round(glove.wmdistance(prep(s1), prep(s4)), 4))

In [ ]:
n = len(sents)
D = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        D[i, j] = glove.wmdistance(prep(sents[i]), prep(sents[j]))
labels = [f"S{i+1}" for i in range(n)]
display(pd.DataFrame(D, index=labels, columns=labels).round(3))

plt.imshow(D, cmap="viridis_r"); plt.colorbar(label="WMD (lower = closer)")
plt.xticks(range(n), labels); plt.yticks(range(n), labels); plt.title("Word Mover's Distance"); plt.show()

In [ ]:
tf = TfidfVectorizer().fit_transform([s1, s2])
print("TF-IDF cosine(s1, s2) =", round(float(cosine_similarity(tf[0], tf[1])[0, 0]), 4), "(no shared words -> 0)")
print("WMD(s1, s2)           =", round(glove.wmdistance(prep(s1), prep(s2)), 4), "(captures semantic closeness)")

In [ ]:
query = "The prime minister talks to journalists in Delhi"
res = sorted((glove.wmdistance(prep(query), prep(s)), s) for s in sents)
for d, s in res: print(round(d, 3), "|", s)